# Estudo Inicial - RAG

Esse documento foi utilizado para entender a estrutura base de como instanciar um RAG, de maneira simples, com duas diferenciações:
1. Instanciando o servidor de maneira local
2. Instanciando o servidor chamando a API do Hugging Face

A base estrutural desse código se baseia em códigos passados pelo prof. Pablo Sampaio, com breves modificações.

Este notebook segue essas etapas:
1. **Configurações iniciais:** Instalações de dependências e configurações do ambiente como chave de acesso do Hugging Face.
2. **Carregando base de conhecimento (KB):** Instancia e separa o documento com base na estruturação de chunking escolhida.
3. **Busca semântica na base de conhecimento:** Utilizando *VectorStoreRetriever*, indicamos como a busca é feita na KB.
4. **RAG:** Construindo o workflow do RAG e um mini teste para observar o funcionamento.

## 1. Configurações Iniciais

Precisamos instalar todas as dependências necessárias para iniciar um RAG, na qual podem ler e compreender PDF's e também que fazem parte da lib **Langchain**

In [ ]:
!pip install -q \
  langchain==1.2.0 \
  langchain-core>=2.0.4 \
  langchain-community>=0.4.1 \
  langchain-google-genai>=4.2.2 \
  langchain-huggingface>=1.2.2 \
  langchain-text-splitters==1.0.0 \
  datasets>=4.8.5 \
  pypdf==4.3.1 \
  pymupdf==1.27.2.3\
  sentence-transformers>=4.4.0 \
  langchain-experimental==0.4.1 \
  langchain-chroma>=1.1.0 \
  langchain-classic>=1.0.5 \
  rank_bm25==0.2.2

Importações necessárias:

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_community.llms import HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever
from langchain_experimental.text_splitter import SemanticChunker
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.retrievers import BaseRetriever
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace, HuggingFaceEndpoint
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sentence_transformers import CrossEncoder

from pydantic import Field, ConfigDict
from langchain_core.documents import Document
from typing import Any, List
import textwrap
import re
import torch



Configurar as chaves de acesso para os modelos:

In [ ]:
from google.colab import userdata
import os

os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get('HuggingFaceKey')


In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HuggingFaceKey'))

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 2. Carregando a Base de conhecimento (KB)

Utilizando a base de conehcimento padrão para os projetos (PDF da biografia do Juscelino Kubistcheck)

In [ ]:
# faz o download do livro pelo notebook python do Colab
!wget https://domainpublic.wordpress.com/wp-content/uploads/2022/10/jk_couto_2ed.pdf

Opcional: Usar alguma forma de limpar o texto com "\n" etc.

In [ ]:
def limpar_texto(texto):
    texto = re.sub(r'­\n', '', texto)      # hifenização invisível
    texto = re.sub(r'-\n', '', texto)       # hifenização normal
    texto = re.sub(r'\d+\s*•\s*JK\s*\n', '', texto)   # "430 • JK"
    texto = re.sub(r'JK\s*•\s*\d+\s*\n', '', texto)   # "JK • 183"
    texto = re.sub(r'(?<!\n)\n(?!\n)', ' ', texto)     # quebras no meio de frase
    texto = re.sub(r' +', ' ', texto)
    return texto.strip()

In [ ]:
arquivo = 'jk_couto_2ed.pdf'
loader = PyMuPDFLoader(f"/content/{arquivo}")
all_pages = loader.load()

for doc in all_pages:
    doc.page_content = limpar_texto(doc.page_content)


print(f"Total de páginas carregadas: {len(all_pages)}")

### 2.1 Separar o Documento (Chunking)

Para a divisão do documento, estamos utilizando a separação semântica, onde utilizamos

1.   Modelos de embeddings, utilizados na decisão de separação semântica
2.   Cálculo da dissimilaridade de contexto em 70%




In [ ]:
embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": device}
)

In [ ]:
text_splitter = SemanticChunker(
    embeddings=embeddings_model,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=70
)

chunks = text_splitter.split_documents(all_pages)
print(f"Documento dividido em {len(chunks)} partes.")

### 2.2 Criar e armazenar esses dados em uma base vetorial

A base vetorial utilizada nesse processo é o ChromaDB, principalmente por ser uma base que integra bem com o Langchain (principal lib que estou usando)

In [ ]:
# 1. Define o banco de dados vetorial (Chroma)
vector_store = Chroma(
    collection_name="embeddings_jk",
    embedding_function=embeddings_model,
    persist_directory="./chroma_db"
)

In [ ]:
# Converte os "chunks" para vetores e os amarzena no vector store
# Retorna os ids dos chunks (mas não vamos precisar deles)
document_ids = vector_store.add_documents(documents=chunks)

## 3 - Busca Híbrida na Base de Conhecimento

A busca é feita de maneira híbrida na base de conhecimento

1.   Busca por BM25, feita procurar principalmente por palavras-chave
2.   Busca por MMR (Maximal Marginal Relevance), Realiza busca por similaridade semântica (embeddings), com penalização de redundância. Retorna resultados relevantes e diversos, evitando chunks muito semelhantes entre si.



In [ ]:
# Quantidade de documentos a serem resgatados por cada busca (padrão = 5 -> total: 10)
quantity = 5

# BM25
bm25 = BM25Retriever.from_documents(chunks, k=quantity)

# Busca semântica por MMR
semantic = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": quantity, "fetch_k": 20, "lambda_mult": 0.6}
)

# Ensemble para juntar os dois
retriever = EnsembleRetriever(
    retrievers=[bm25, semantic],
    weights=[0.5, 0.5]  # peso igual, visto que os nomes próprios e siglas são frequentes em biografia
)

In [ ]:
query = "Qual a origem da família de Juscelino Kubitschek? Eles vieram de qual país e eram de qual etnia?"

# Agora, basta chamar "invoke()" para fazer a busca, sempre usando os parâmetros acima
retrieved_docs = retriever.invoke(query)
print("Chunks recuperados:", len(retrieved_docs))

In [ ]:
print("Texto do 1o chunk recuperado:")
retrieved_docs[0].page_content[:300]

## 3.1 Reranking - Utilização de CrossEncoder

Um modelo cross-encoder reavalia os documentos retornados, atribuindo uma pontuação para cada par (query, documento).


> Um cross-encoder é um tipo de modelo usado para medir o quão relevante um texto é para outro, normalmente uma pergunta e um documento.


Os resultados são então ordenados por relevância, retornando apenas os mais bem classificados, com maior precisão semântica.

In [ ]:
# Aceita chunks em português
cross_encoder = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1')

class CrossEncoderRetriever(BaseRetriever):
    vectorstore: Any
    cross_encoder: Any
    k: int = Field(default=10)
    rerank_top_k: int = Field(default=3)

    model_config = ConfigDict(
        arbitrary_types_allowed=True
    )

    def _get_relevant_documents(self, query: str, *, run_manager) -> List[Document]:
        # 1. Retriever inicial, feito com o Ensemble previamente programado
        initial_docs = retriever.invoke(query)

        # 2. Prepara os pares com: query + chunk para calcular a pontuação (reranking)
        pairs = [[query, doc.page_content] for doc in initial_docs]

        # 3. Pega a pontuação calculada
        scores = self.cross_encoder.predict(pairs)

        # 4. Ordena pela pontuação
        scored_docs = sorted(zip(initial_docs, scores), key=lambda x: x[1], reverse=True)

        return [doc for doc, _ in scored_docs[:self.rerank_top_k]]

    async def aget_relevant_documents(self, query: str) -> List[Document]:
        raise NotImplementedError("Async retrieval not implemented")

In [ ]:
cross_encoder_retriever = CrossEncoderRetriever(
    vectorstore=vector_store,
    cross_encoder=cross_encoder,
    k=quantity*2,  # Retorna a quantidade * 2
    rerank_top_k=3  # Depois de tudo, vai retornar os top 3 melhores
)

## 4 RAG

### 4.1 Prompt e Modelo

In [ ]:
# Este será o prompt usado para gerar respostas, ainda embrionário
prompt = PromptTemplate(
    input_variables=["contexto", "pergunta"],
    template="""
Você é um assistente que responde perguntas de múltipla escolha com base EXCLUSIVAMENTE nos trechos abaixo.

# Trechos do documento
{contexto}

# Pergunta
{pergunta}

# Como responder
Passo 1: Identifique qual trecho contém informação diretamente relacionada à pergunta, prestando atenção às datas e nomes mencionados.
Passo 2: Ignore trechos que falem de eventos em datas diferentes da pergunta.
Passo 3: Com base apenas no trecho correto, escolha a alternativa certa.
Passo 4: Justifique o motivo da resposta de maneira extensa e bem explicativa, pode utilizar um parágrafo.
Passo 5: Quais fatos do contexto são relevantes para a pergunta?
Passo 6: Existe alguma relação de oposição, causalidade ou sequência temporal entre os elementos?
Passo 7: Qual alternativa é diretamente suportada pelo contexto?
Passo 8: Use APENAS informações presentes literalmente no contexto

# Resposta final
"""
)

### 4.2 Chamadas da LLM

> **ESCOLHA UM OU OUTRO**


#### 4.2.1 Chamada local da LLM [Mais lento]

In [ ]:
MODEL_ID = "google/gemma-3-4b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    dtype="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    max_length=None,
    do_sample=True,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=pipe)

#### 4.2.2 Chamada via HuggingFaceEndpoint (API) [Mais rápido]

In [ ]:
llm = HuggingFaceEndpoint(
        repo_id="google/gemma-3-4b-it",
        task="text-generation",
        max_new_tokens=512,
        do_sample=True,
    )
    # Instancia uma classe do LangChain que permite acessar modelos do HF
llm = ChatHuggingFace(llm=llm)

### 4.3 Definição de workflow

1. Busca semântica
2. Re-ranking com CrossEncoder
3. (Opcional) Mostra os melhores chunks recuperados
4. Montagem do prompt com base no que foi recuperado
5. Gera a resposta chamando o LLM

In [ ]:
def rag_workflow(pergunta_usuario, mostrar_chunks=False):
    # 1. Recuperação (busca semântica)
    docs = cross_encoder_retriever.invoke(pergunta_usuario)

    # (Extra): mostra o início dos chuncks recuperados
    if mostrar_chunks:
        print("----")
        for i, doc in enumerate(docs):
            print(" => chunk", i+1, ":", doc.page_content[:80], "...")
        print("----")

    # 2. Montagem do prompt
    # 2.1 - une todo o contexto em uma string só
    full_context = "\n---\n".join(d.page_content for d in docs)

    # 2.2 - aplica o template de prompt
    final_prompt = prompt.format(
        contexto=full_context,
        pergunta=pergunta_usuario
    )

    # 3. Gera a resposta chamando o LLM
    response = llm.invoke(final_prompt)

    content = response.content if hasattr(response, 'content') else response
    return content, [d.page_content for d in docs]

In [ ]:
question = """
A trágica quebra institucional do golpe civil-militar de 1964 interrompeu a sobrevida política de JK, situação que atingiu o ápice com a dolorosa cassação do seu mandato de senador e suspensão severa de seus direitos políticos em 8 de junho de 1964. Como o autor descreve detalhadamente a mecânica de pressão política e militar que emparedou o governo de Humberto Castello Branco e forçou a punição sumária do ex-presidente?
A) O presidente Castello Branco, motivado intimamente por um repúdio ideológico cultivado durante toda a sua carreira, exigiu e redigiu o decreto de expurgo de forma autoritária e despótica, isolando-se das deliberações conjuntas do Conselho de Segurança Nacional Agindo por vontade unânime própria, o general argumentou, baseando-se em convicções irrevogáveis e anunciando publicamente na TV, que Juscelino era o orquestrador financeiro da guerrilha comunista que tomou conta das praças de Belo Horizonte.
B) O afastamento compulsório e dramático de Juscelino foi exigido de forma quase intempestiva e veemente pelo ministro da Guerra, o general Costa e Silva, atuando sob forte clamor dos oficiais da "linha dura" e lacerdistas, os quais consideravam o senador o pior e maior entrave oculto da Revolução em decorrência de seu formidável favoritismo e projeção para as futuras eleições presidenciais. Pressionado impiedosamente por esse flanco radical militar e confrontado por um dossiê calhamaço com suspeitas insatisfatórias de corrupção pessoal, o ponderado Castello Branco cedeu à contingência e executou burocraticamente a condenação.
C) Após conversações diplomáticas madrugadoras travadas discretamente num requintado apartamento em Copacabana, Juscelino optou por solicitar oficialmente ao general Castello Branco a suspensão abrandada dos seus próprios direitos eleitorais. Ao prever um insustentável derramamento de sangue que atingiria a oposição de sua época, JK aceitou migrar num exílio protegido até Paris, exigindo como única moeda de troca para o abandono político o direito de seu PSD figurar oficialmente na estruturação legal e militar do novo governo instituído e assumir postos nos ministérios da Defesa.
D) Uma vigorosa articulação originada dentro dos gabinetes do Supremo Tribunal Federal, sob influência das intensas e assustadoras passeatas que cobravam probidade no país, elaborou e aplicou uma decisão unânime limitadora do mandato e imunidade de Juscelino. O poder Judiciário considerou devidamente provadas todas as transações escusas do político mineiro com embaixadores ideológicos em Havana, coagindo moral e legalmente um presidente Castello Branco hesitante a endossar o afastamento do líder como uma determinação inquestionável e constitucional da Justiça pátria.
"""

In [ ]:
resposta1,recovered_chunks = rag_workflow(question,
                         mostrar_chunks=True)

print("Saída Final:\n")
resposta_em_linhas = textwrap.fill(resposta1, width=90)
print(resposta_em_linhas)

In [ ]:
recovered_chunks

# 5 Avaliar a resposta da IA (Utilizando LLM-as-Judge) *Opcional*
Para testar, utilizamos o RAGAS, uma lib open-source para testar e dar evaluate em RAG's. Site: https://www.ragas.io/

Não possuimos uma etapa muito aprofundada, somente para ter um nível de análise do RAG

In [ ]:
!pip install ragas
!pip install datasets

In [ ]:
test_set = [
    # Tópico 1: Conteúdo Presente
    (
        "Qual foi o binômio em que Juscelino Kubitschek baseou grande parte do seu plano de desenvolvimento ao assumir o governo de Minas Gerais e a Presidência da República?\n"
        "A) Saúde e Educação\n"
        "B) Agricultura e Mineração\n"
        "C) Energia e Transportes\n"
        "D) Habitação e Segurança",
        "C) Energia e Transportes"
    ),
    (
        "Como JK classificava a construção de Brasília dentro do seu Programa de Metas?\n"
        "A) Meta Provisória\n"
        "B) Meta-síntese\n"
        "C) Meta de Ocupação\n"
        "D) Meta Secundária",
        "B) Meta-síntese"
    ),
    (
        "Em que data exata o governo militar assinou e divulgou o decreto que cassou o mandato de senador e suspendeu os direitos políticos de Juscelino Kubitschek?\n"
        "A) 10 de novembro de 1937\n"
        "B) 24 de agosto de 1954\n"
        "C) 8 de junho de 1964\n"
        "D) 13 de dezembro de 1968",
        "C) 8 de junho de 1964"
    ),

    # Tópico 2: Capítulo 8 (Prefeito de Belo Horizonte)
    (
        "Qual apelido Juscelino Kubitschek ganhou da população devido ao ritmo acelerado de obras e intervenções urbanas em sua gestão na prefeitura de Belo Horizonte?\n"
        "A) Prefeito Bossa Nova\n"
        "B) Prefeito Furacão\n"
        "C) Prefeito a Jato\n"
        "D) Prefeito Construtor",
        "B) Prefeito Furacão"
    ),
    (
        "Qual obra urbanística inovadora e marco da arquitetura moderna foi construída por JK na prefeitura de Belo Horizonte sem o conhecimento prévio do governador Benedito Valadares?\n"
        "A) A construção do Palácio das Mangabeiras\n"
        "B) A canalização do ribeirão do Inferno\n"
        "C) O conjunto arquitetônico do bairro da Pampulha\n"
        "D) O asfaltamento da avenida Afonso Pena",
        "C) O conjunto arquitetônico do bairro da Pampulha"
    ),
    (
        "Como Juscelino Kubitschek foi comunicado de sua nomeação surpreendente para a prefeitura de Belo Horizonte em abril de 1940?\n"
        "A) Ele venceu uma eleição direta contra o candidato da UDN\n"
        "B) Ele recebeu um convite formal do presidente Getúlio Vargas no Rio de Janeiro\n"
        "C) Pela convocação do diretor da Imprensa Oficial, que lhe mostrou a ordem para publicar o decreto já assinado no diário oficial 'Minas Gerais'\n"
        "D) Ele assumiu interinamente após a morte súbita do prefeito anterior",
        "C) Pela convocação do diretor da Imprensa Oficial, que lhe mostrou a ordem para publicar o decreto já assinado no diário oficial 'Minas Gerais'"
    ),

    # Tópico 3: Sobre uma Página/Trecho Específico (O acidente e a morte)
    (
        "Em que veículo Juscelino Kubitschek viajava no desastre de 1976?\n"
        "A) Em um monomotor Bonanza\n"
        "B) Em uma carreta Scania-Vabis\n"
        "C) Em um táxi aéreo\n"
        "D) Em um Chevrolet Opala 1970",
        "D) Em um Chevrolet Opala 1970"
    ),
    (
        "Quem era o motorista de JK no acidente?\n"
        "A) Ladislau Borges\n"
        "B) Josias Nunes de Oliveira\n"
        "C) Geraldo Ribeiro\n"
        "D) Bernardo Sayão",
        "C) Geraldo Ribeiro"
    ),
    (
        "Qual a data exata do falecimento de JK?\n"
        "A) 24 de agosto de 1954\n"
        "B) 22 de agosto de 1976\n"
        "C) 12 de setembro de 1902\n"
        "D) 8 de junho de 1964",
        "B) 22 de agosto de 1976"
    ),
    (
        "Onde ocorreu o trágico desastre automobilístico que vitimou Juscelino Kubitschek?\n"
        "A) No quilômetro 165 da Rodovia Belém-Brasília\n"
        "B) Na curva do quilômetro 165 da Via Dutra, município de Resende\n"
        "C) Na estrada de terra de acesso à sua fazenda em Luziânia\n"
        "D) No aeroporto de Congonhas, em São Paulo",
        "B) Na curva do quilômetro 165 da Via Dutra, município de Resende"
    ),
    (
        "Segundo as informações da perícia técnica oficial, o que teria feito o carro de JK se desgovernar antes de bater de frente com a carreta Scania-Vabis?\n"
        "A) Um pneu que estourou devido à pista molhada\n"
        "B) Um toque na traseira esquerda dado por um ônibus de passageiros da Viação Cometa\n"
        "C) Uma falha nos freios do veículo\n"
        "D) O excesso de velocidade ao tentar desviar de um caminhão",
        "B) Um toque na traseira esquerda dado por um ônibus de passageiros da Viação Cometa"
    )
]

In [ ]:
!pip install langchain-google-genai

In [ ]:
from datasets import Dataset

questions = []
answers = []
contexts = []
ground_truths = []

for query, truth in test_set:
    # Captura a resposta e os chunks do seu workflow
    # IMPORTANTE: Ajuste seu workflow para retornar (resposta, lista_de_chunks)
    res_texto, chunks_lista = rag_workflow(query, mostrar_chunks=False)

    questions.append(query)
    answers.append(res_texto)
    contexts.append(chunks_lista) # Deve ser uma lista de strings
    ground_truths.append(truth)

    print(f"✅ Pergunta processada: {query[:50]}...")

# Criando o Dataset final
data_dict = {
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths
}
dataset = Dataset.from_dict(data_dict)

In [ ]:
!pip install -U ragas langchain-core langchain-community langchain-google-genai datasets --quiet

In [ ]:
import os
import nest_asyncio
from ragas import evaluate, RunConfig # Importação direta
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from langchain_huggingface import HuggingFaceEndpoint, HuggingFaceEmbeddings

nest_asyncio.apply()

# 2. Inicializa o LLM Juiz (Via Inference API - Gratuito/Serverless)
# Usando o Llama-3-8B-Instruct ou Mistral-7B
hf_llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V4-Pro",
    task="text-generation",
    temperature=0.01,
)

config = RunConfig(
    max_workers=1,       # Reduz para apenas 2 tarefas paralelas (evita gargalo)
    timeout=100,          # Aumenta o tempo limite para 60 segundos por job
)

ragas_llm = LangchainLLMWrapper(hf_llm)

# 3. Inicializa o Embedding (Local - roda na sua RAM/GPU)
# O modelo 'BAAI/bge-small-en-v1.5' ou 'sentence-transformers/all-MiniLM-L6-v2'
hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
ragas_emb = LangchainEmbeddingsWrapper(hf_embeddings)

# 4. Configura as métricas
metrics = [faithfulness, answer_relevancy, context_precision]

for m in metrics:
    m.llm = ragas_llm
    if hasattr(m, 'embeddings'):
        m.embeddings = ragas_emb

# 5. Executa a avaliação
result = evaluate(
    dataset.select([0,1]),
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_emb,
    run_config=config
)

print(result)

In [ ]:
result
